# Information-aware gates for overlapping spatial sources

Events have two measured coordinates and arise from two overlapping intensity components. We want a fixed sixteen-bin interface for estimating the two component coefficients, not a faithful reconstruction of spatial density.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as sq
from examples.synthetic_problems import spatial_sources

problem = spatial_sources()
train, validation, test = problem.train, problem.validation, problem.test
train.observations.shape, train.scores.shape

## Exploratory view

Color shows one score coordinate: local sensitivity to the first coefficient. Geometric proximity and inferential similarity are related here, but they are not the same objective.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
density = axes[0].scatter(
    train.observations[:, 0], train.observations[:, 1], c=train.weights, s=5, cmap="viridis"
)
axes[0].set(title="Reference intensity", xlabel="x", ylabel="y")
fig.colorbar(density, ax=axes[0])
score = axes[1].scatter(
    train.observations[:, 0], train.observations[:, 1], c=train.scores[:, 0], s=5, cmap="coolwarm"
)
axes[1].set(title="Score for coefficient 1", xlabel="x", ylabel="y")
fig.colorbar(score, ax=axes[1]);

## Fit in score space, predict in score space

The fitted result always predicts in the same representation used for fitting. Here the application has already evaluated scores, so both `fit_quantizer(ScoreSample(...))` and `predict` receive two-column score matrices.

In [ ]:
quantizer = sq.fit_quantizer(
    sq.ScoreSample(train.scores, train.weights),
    validation=sq.ScoreSample(validation.scores, validation.weights),
    n_bins=16,
    criterion=sq.DOptimality(),
    config=sq.SoftVoronoiConfig(seed=42, initializer_restarts=3, max_steps=60, record_every=10),
)
labels = np.asarray(quantizer.predict_scores(test.scores))
report = quantizer.evaluate_scores(test.scores, test.weights)
np.bincount(labels, minlength=16), report.geometric_mean_retention

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(test.observations[:, 0], test.observations[:, 1], c=labels, cmap="tab20", s=5)
ax.set(title="Frozen hard bins in the physical plane", xlabel="x", ylabel="y");

## What to check next

Retention measures compression of the supplied local score model. A production analysis should additionally validate that model away from the reference point, inspect occupancy, and test the complete downstream count likelihood for bias and coverage.